# 04 · Eleição do vencedor e promoção controlada

1. **Vencedor do ciclo**: menor MAE na validação fixa (em caso de empate, vence o Ridge, que é mais simples).
2. **Confronto com o vigente**: o modelo vigente é recarregado (com seu pré-processamento e suas features) e reavaliado **na mesma validação fixa**.
3. **Regra de promoção (pré-definida)**: o desafiante precisa (a) superar o baseline e (b) ter MAE **pelo menos 2% menor** que o do vigente. Se não houver vigente, o primeiro modelo válido é promovido.
4. Caso contrário, **o vigente é mantido**. Os modelos anteriores ficam preservados em `models/historico/`.

In [1]:
import sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import warnings
warnings.filterwarnings("ignore")
import json
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image

from src.utils import preparar_ambiente, carregar_config, carregar_contexto, salvar_contexto, dir_evidencias, configurar_mlflow
preparar_ambiente()
cfg = carregar_config()
pd.set_option("display.width", 160)

In [2]:
from src.promocao import (eleger_vencedor, avaliar_vigente, decidir, promover, salvar_manifesto,
                          carregar_manifesto, registrar_historico)
from src.ingestao import carregar_validacao

ctx = carregar_contexto()
ev = dir_evidencias(ctx["cycle_id"])
res = ctx["resultados"]

# Verificações de integridade: falha aqui interrompe o workflow e impede promoção indevida
assert set(res) == {"A", "B"}, "Os dois candidatos precisam ter sido treinados"
mlflow = configurar_mlflow(cfg)
for k, r in res.items():
    tags = mlflow.get_run(r["run_id"]).data.tags
    assert tags["cycle_id"] == ctx["cycle_id"] and tags["data_version"] == ctx["data_version"], f"Run {k} de outro ciclo"
    assert mlflow.get_run(r["run_id"]).data.params["features"] == ",".join(ctx["features_eleitas"]), "Features diferentes"

venc = eleger_vencedor(res)
r_v = res[venc]
validacao = carregar_validacao(cfg)
manifesto_ant, met_vig = avaliar_vigente(validacao, cfg)
mae_vig = None if met_vig is None else met_vig["mae"]
promove, motivo = decidir(r_v["val_mae"], mae_vig, ctx["baseline_val_mae"], cfg)
decisao = "promovido" if promove else "mantido"

print(f"Vencedor do ciclo: modelo {venc} ({r_v['algoritmo']}) – MAE val {r_v['val_mae']:.3f}")
print("Vigente atual:", "nenhum" if manifesto_ant is None else f"{manifesto_ant['algoritmo']} do ciclo {manifesto_ant['cycle_id']} – MAE val reavaliado {mae_vig:.3f}")
print(f"DECISÃO: {decisao.upper()} – {motivo}")

Vencedor do ciclo: modelo B (random_forest) – MAE val 5.784
Vigente atual: random_forest do ciclo c20261124T225921Z-local – MAE val reavaliado 5.728
DECISÃO: MANTIDO – melhoria de -1.0% < 2% exigidos (vigente 5.728 x desafiante 5.784)


In [3]:
info_ciclo = {
    "cycle_id": ctx["cycle_id"], "data_version": ctx["data_version"], "algoritmo": r_v["algoritmo"],
    "candidato": venc, "features": ctx["features_eleitas"], "metrica": cfg["avaliacao"]["metrica_principal"],
    "val_mae": r_v["val_mae"], "val_rmse": r_v["val_rmse"], "val_r2": r_v["val_r2"], "run_id": r_v["run_id"],
    "mlflow_model_uri": f"runs:/{r_v['run_id']}/modelo",
}
if promove:
    manifesto = promover(r_v["arquivo"], {**info_ciclo, "decisao": "promovido", "motivo": motivo})
else:
    manifesto = carregar_manifesto()
manifesto["ultima_avaliacao"] = {"cycle_id": ctx["cycle_id"], "data_version": ctx["data_version"], "decisao": decisao,
                                 "motivo": motivo, "desafiante": r_v["algoritmo"], "mae_desafiante": r_v["val_mae"],
                                 "mae_vigente_reavaliado": mae_vig, "run_id_desafiante": r_v["run_id"]}
salvar_manifesto(manifesto)

decisao_json = {"cycle_id": ctx["cycle_id"], "data_version": ctx["data_version"], "vencedor": venc,
                "vencedor_algoritmo": r_v["algoritmo"], "mae_vencedor": round(r_v["val_mae"], 4),
                "mae_perdedor": round(res["B" if venc == "A" else "A"]["val_mae"], 4),
                "mae_vigente_antes": None if mae_vig is None else round(mae_vig, 4),
                "baseline_val_mae": round(ctx["baseline_val_mae"], 4), "decisao": decisao, "motivo": motivo,
                "vigente_apos": {"cycle_id": manifesto["cycle_id"], "algoritmo": manifesto["algoritmo"], "val_mae": manifesto["val_mae"]}}
(ev / "decisao.json").write_text(json.dumps(decisao_json, indent=2, ensure_ascii=False), encoding="utf-8")

registrar_historico({
    "cycle_id": ctx["cycle_id"], "gatilho": ctx["gatilho"], "lote": ctx["lote"] or "-", "data_version": ctx["data_version"],
    "n_treino": ctx["n_treino"], "features_eleitas": "|".join(ctx["features_eleitas"]),
    "mae_ridge": round(res["A"]["val_mae"], 4), "mae_random_forest": round(res["B"]["val_mae"], 4),
    "vencedor": r_v["algoritmo"], "mae_vigente_antes": None if mae_vig is None else round(mae_vig, 4),
    "decisao": decisao, "vigente_apos": f"{manifesto['algoritmo']}@{manifesto['cycle_id']}",
    "drift_alerta": ctx.get("drift_alerta", False),
})

with mlflow.start_run(run_name=f"decisao_{ctx['cycle_id']}"):
    mlflow.set_tags({"cycle_id": ctx["cycle_id"], "data_version": ctx["data_version"], "etapa": "decisao", "decisao": decisao})
    mlflow.log_params({"vencedor": r_v["algoritmo"], "regra_promocao": f"melhoria>={cfg['promocao']['melhoria_minima_relativa']} e melhor que baseline",
                       "motivo": motivo[:400], "vigente_apos": f"{manifesto['algoritmo']}@{manifesto['cycle_id']}"})
    mlflow.log_metrics({"mae_vencedor": r_v["val_mae"], "promovido": int(promove), **({"mae_vigente_antes": mae_vig} if mae_vig else {})})
    mlflow.log_dict(manifesto, "decisao/modelo_vigente.json")
for k, r in res.items():
    mlflow.MlflowClient().set_tag(r["run_id"], "resultado", ("promovido" if promove else "venceu_mas_nao_promovido") if k == venc else "perdeu")

salvar_contexto({"decisao": decisao, "motivo": motivo})
print(json.dumps(decisao_json, indent=2, ensure_ascii=False))

{
  "cycle_id": "c20261124T230002Z-local",
  "data_version": "v3-c6be3d56",
  "vencedor": "B",
  "vencedor_algoritmo": "random_forest",
  "mae_vencedor": 5.7836,
  "mae_perdedor": 6.1568,
  "mae_vigente_antes": 5.7281,
  "baseline_val_mae": 14.0999,
  "decisao": "mantido",
  "motivo": "melhoria de -1.0% < 2% exigidos (vigente 5.728 x desafiante 5.784)",
  "vigente_apos": {
    "cycle_id": "c20261124T225921Z-local",
    "algoritmo": "random_forest",
    "val_mae": 5.728106157267119
  }
}
